In [ ]:
import requests
import os
import time
from urllib.parse import quote

# TMDb API key and the number of actor images to download
API_KEY = "7ade8bf5eb9b1fac6823c5b57bfa7e5a"
NUM_ACTORS = 100 

# Create 'dataset' directory if it doesn't exist
os.makedirs("dataset", exist_ok=True)

# Function to fetch popular people from TMDb API
def get_popular_people(api_key, page=1):
    # Construct the API URL for fetching popular people
    url = f"https://api.themoviedb.org/3/person/popular?api_key={api_key}&page={page}"
    response = requests.get(url)
    # Return the list of popular people from the API response
    return response.json().get("results", [])

# Function to download the profile image of a person
def download_profile_image(person):
    # Get the name and profile image path of the person
    name = person["name"].replace("/", "_")
    profile_path = person.get("profile_path")
    # Skip if the person doesn't have a profile image
    if not profile_path:
        return False
    # Construct the image URL and download the image
    image_url = f"https://image.tmdb.org/t/p/w500{profile_path}"
    image_data = requests.get(image_url).content
    # Save the image to the 'dataset' directory
    with open(f"dataset/{quote(name)}.jpg", "wb") as f:
        f.write(image_data)
    return True

# Download images until the specified number of actors is reached
downloaded, page = 0, 1
while downloaded < NUM_ACTORS:
    # Fetch popular people from the current page
    people = get_popular_people(API_KEY, page)
    if not people:
        break
    for person in people:
        # Stop if the required number of images is downloaded
        if downloaded >= NUM_ACTORS:
            break
        if download_profile_image(person):
            downloaded += 1
        time.sleep(0.2)
    page += 1
